# Lesson 3.2 — ADQL Fundamentals

## Learning Objectives
- Write ADQL queries using SELECT, FROM, WHERE, ORDER BY, and LIMIT
- Filter transactions by name, response time, user experience, and error state
- Use comparison operators and string functions in ADQL
- Validate queries in the AppDynamics UI and via the Analytics REST API

**Reference**: [ADQL Reference](https://help.splunk.com/en/appdynamics-saas/analytics/26.2.0/analytics/adql-reference/adql-queries) | [ADQL Queries in Dash Studio](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/dashboards-and-reports/dash-studio/data-binding/adql-queries)

## 1. ADQL vs SQL

ADQL (AppDynamics Query Language) is a **SQL-like** query language designed for querying AppDynamics Analytics data.

If you know SQL, ADQL will be familiar. Key differences:

| Aspect | SQL | ADQL |
|--------|-----|------|
| **Time range** | Part of the query (WHERE clause) or external | Controlled by the **time range selector** (not in query) |
| **Tables** | Database tables | Fixed: `transactions`, `browser_records`, etc. |
| **JOINs** | Supported | Not supported in ADQL |
| **Subqueries** | Supported | Not supported in ADQL |
| **Aggregations** | GROUP BY | Works similarly in ADQL |
| **Series** | N/A | Special `series()` function for time-series output |
| **Syntax** | Varies by DB | Similar to standard SQL but with AppD-specific functions |

**Basic ADQL query structure**:
```sql
SELECT <fields_or_aggregations>
FROM <table>
[WHERE <conditions>]
[ORDER BY <field> [ASC|DESC]]
[LIMIT <n>]
```

In [ ]:
# We'll use our simulated records from Lesson 3.1 to practice ADQL logic
# These represent what AppDynamics stores in the Events Service

import random, json
from datetime import datetime, timedelta

random.seed(42)

def generate_transaction_dataset(n=500) -> list:
    records = []
    transactions = ["/checkout", "/search", "/cart/add", "/cart/view", "/login", "/product/{id}"]
    tiers = ["WebFrontend", "MobileAPI"]
    base_ts = int((datetime.now() - timedelta(minutes=30)).timestamp() * 1000)
    
    for i in range(n):
        tx = random.choice(transactions)
        ue = random.choices(["NORMAL","SLOW","VERY_SLOW","ERROR"], weights=[82,10,5,3])[0]
        rt_map = {"NORMAL": random.gauss(280,60), "SLOW": random.gauss(900,100),
                  "VERY_SLOW": random.gauss(2100,300), "ERROR": random.gauss(500,200)}
        records.append({
            "eventTimestamp": base_ts + i * 3600,
            "transactionName": tx,
            "tierName": random.choice(tiers),
            "applicationName": "StoreFront",
            "responseTime": max(50, int(rt_map[ue])),
            "userExperience": ue,
            "errorOccurred": ue == "ERROR",
            "userId": str(random.randint(1000, 9999)),
            "country": random.choice(["United States","Germany","Japan","Brazil","United Kingdom"]),
            "browser": random.choice(["Chrome","Firefox","Safari","Edge"]),
            "deviceType": random.choice(["Desktop","Mobile","Tablet"]),
        })
    return records

DATA = generate_transaction_dataset(500)
print(f"Dataset: {len(DATA)} transaction records")
print("Sample record:")
print(json.dumps(DATA[0], indent=2))

In [ ]:
# Helper: mini ADQL simulator on our dataset
# This lets us practice ADQL logic in Python before running on a real controller.

def adql_filter(data, conditions):
    """Apply a list of (field, operator, value) conditions to filter records."""
    result = data
    for field, op, val in conditions:
        if op == '=':
            result = [r for r in result if r.get(field) == val]
        elif op == '!=':
            result = [r for r in result if r.get(field) != val]
        elif op == '>':
            result = [r for r in result if r.get(field, 0) > val]
        elif op == '>=':
            result = [r for r in result if r.get(field, 0) >= val]
        elif op == '<':
            result = [r for r in result if r.get(field, 0) < val]
        elif op == 'IN':
            result = [r for r in result if r.get(field) in val]
        elif op == 'LIKE':
            import fnmatch
            pattern = val.replace('%', '*').replace('_', '?')
            result = [r for r in result if fnmatch.fnmatch(str(r.get(field,'')), pattern)]
    return result

## 2. Basic SELECT Queries

### Query 1: Select all transactions (first 10)
```sql
SELECT * FROM transactions LIMIT 10
```

### Query 2: Select specific fields
```sql
SELECT transactionName, responseTime, userExperience FROM transactions LIMIT 10
```

### Query 3: Filter by transaction name
```sql
SELECT * FROM transactions WHERE transactionName = '/checkout' LIMIT 20
```

### Query 4: Filter by response time
```sql
SELECT transactionName, responseTime, userId FROM transactions
WHERE responseTime > 2000
ORDER BY responseTime DESC
LIMIT 50
```

In [ ]:
# Simulate ADQL Query 3: Find all /checkout transactions
# ADQL: SELECT transactionName, responseTime, userExperience FROM transactions
#        WHERE transactionName = '/checkout'
#        ORDER BY responseTime DESC LIMIT 10

checkout_txns = adql_filter(DATA, [("transactionName", "=", "/checkout")])
checkout_txns.sort(key=lambda r: r["responseTime"], reverse=True)

print("ADQL: SELECT transactionName, responseTime, userExperience FROM transactions")
print("      WHERE transactionName = '/checkout'")
print("      ORDER BY responseTime DESC LIMIT 10")
print()
print(f"{'transactionName':<20} {'responseTime':>14} {'userExperience':<15}")
print("-" * 55)
for r in checkout_txns[:10]:
    print(f"{r['transactionName']:<20} {r['responseTime']:>11}ms   {r['userExperience']:<15}")
print(f"\n(Total /checkout transactions: {len(checkout_txns)})")

In [ ]:
# Simulate ADQL Query 4: Find the slowest transactions (> 2000ms)
# ADQL: SELECT transactionName, responseTime, userId, country FROM transactions
#        WHERE responseTime > 2000
#        ORDER BY responseTime DESC
#        LIMIT 10

slow_txns = adql_filter(DATA, [("responseTime", ">", 2000)])
slow_txns.sort(key=lambda r: r["responseTime"], reverse=True)

print("ADQL: SELECT transactionName, responseTime, userId, country FROM transactions")
print("      WHERE responseTime > 2000 ORDER BY responseTime DESC LIMIT 10")
print()
print(f"{'transactionName':<20} {'responseTime':>14} {'userId':>8} {'country':<20}")
print("-" * 68)
for r in slow_txns[:10]:
    print(f"{r['transactionName']:<20} {r['responseTime']:>11}ms   {r['userId']:>8} {r['country']:<20}")

## 3. WHERE Clause Operators

ADQL supports standard comparison and string operators:

```sql
-- Comparison operators
WHERE responseTime > 1000
WHERE responseTime >= 500 AND responseTime <= 2000
WHERE userExperience != 'NORMAL'

-- IN clause
WHERE userExperience IN ('SLOW', 'VERY_SLOW', 'ERROR')
WHERE transactionName IN ('/checkout', '/cart/add')

-- String matching
WHERE transactionName CONTAINS '/cart'
WHERE transactionName STARTS WITH '/api'

-- Boolean
WHERE errorOccurred = true

-- NULL check
WHERE userId IS NOT NULL

-- Compound conditions
WHERE transactionName = '/checkout' AND responseTime > 5000 AND errorOccurred = false
WHERE userExperience = 'ERROR' OR responseTime > 3000
```

> **Note**: String values in ADQL are case-sensitive unless using `CONTAINS` or `STARTS WITH`.

In [ ]:
# Simulate ADQL Query 5: Find all errors in any transaction
# ADQL: SELECT transactionName, responseTime, userId, country, browser
#        FROM transactions WHERE errorOccurred = true
#        ORDER BY eventTimestamp DESC LIMIT 20

errors = adql_filter(DATA, [("errorOccurred", "=", True)])
errors.sort(key=lambda r: r["eventTimestamp"], reverse=True)

print("ADQL: SELECT transactionName, responseTime, userId, country, browser")
print("      FROM transactions WHERE errorOccurred = true")
print("      ORDER BY eventTimestamp DESC LIMIT 20")
print()
print(f"{'Transaction':<20} {'RT (ms)':>8} {'UserId':>8} {'Country':<20} {'Browser':<12}")
print("-" * 74)
for r in errors[:20]:
    print(f"{r['transactionName']:<20} {r['responseTime']:>8} {r['userId']:>8} {r['country']:<20} {r['browser']:<12}")
print(f"\nTotal errors: {len(errors)} out of {len(DATA)} ({len(errors)/len(DATA)*100:.1f}%)")

In [ ]:
# Simulate ADQL Query 6: Multi-condition filter
# Find slow /checkout transactions that are NOT errors
# ADQL: SELECT transactionName, responseTime, userExperience, userId
#        FROM transactions
#        WHERE transactionName = '/checkout'
#          AND userExperience IN ('SLOW', 'VERY_SLOW')
#          AND errorOccurred = false
#        ORDER BY responseTime DESC
#        LIMIT 10

slow_checkout_no_error = adql_filter(DATA, [
    ("transactionName", "=", "/checkout"),
    ("userExperience",  "IN", {"SLOW", "VERY_SLOW"}),
    ("errorOccurred",   "=", False),
])
slow_checkout_no_error.sort(key=lambda r: r["responseTime"], reverse=True)

print("ADQL: SELECT transactionName, responseTime, userExperience, userId")
print("      FROM transactions")
print("      WHERE transactionName = '/checkout'")
print("        AND userExperience IN ('SLOW', 'VERY_SLOW')")
print("        AND errorOccurred = false")
print("      ORDER BY responseTime DESC LIMIT 10")
print()
for r in slow_checkout_no_error[:10]:
    print(f"  {r['transactionName']:<20} {r['responseTime']:>7}ms  {r['userExperience']:<10}  User: {r['userId']}")

In [ ]:
# EXERCISE 3.2: Write ADQL queries for these business questions
#
# For each question, write the ADQL query as a comment,
# then implement the equivalent Python filter logic.

print("=== Question 1: Which users had errors in Germany? ===")
# ADQL: SELECT userId, transactionName, responseTime FROM transactions
#        WHERE country = 'Germany' AND errorOccurred = true LIMIT 20

# TODO: Implement the filter
q1_result = None  # replace with adql_filter(...)

if q1_result is not None:
    print(f"Found {len(q1_result)} German error transactions")
    for r in q1_result[:5]:
        print(f"  User {r['userId']}: {r['transactionName']} - {r['responseTime']}ms")
print()

print("=== Question 2: What's the slowest transaction from Mobile users? ===")
# ADQL: SELECT transactionName, responseTime, userId FROM transactions
#        WHERE deviceType = 'Mobile' ORDER BY responseTime DESC LIMIT 1

# TODO: Implement the filter
q2_result = None

if q2_result is not None and len(q2_result) > 0:
    r = q2_result[0]
    print(f"Slowest mobile transaction: {r['transactionName']} - {r['responseTime']}ms (User: {r['userId']})")
print()

print("=== Question 3: How many Safari errors occurred? ===")
# ADQL: SELECT count(*) FROM transactions WHERE browser = 'Safari' AND errorOccurred = true

# TODO: Implement the filter
q3_count = 0  # replace with actual count
print(f"Safari errors: {q3_count}")

## Summary

- ADQL is SQL-like: `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT`
- Key tables: `transactions`, `browser_records`, `application_logs`
- The time range is controlled by the **UI time picker**, not the query
- Operators: `=`, `!=`, `>`, `>=`, `<`, `<=`, `IN`, `CONTAINS`, `STARTS WITH`, `IS NULL`
- Compound conditions: `AND`, `OR`, `NOT`
- String values are **case-sensitive** in most operators

---

**Next**: [Lesson 3.3 — ADQL Aggregations and Time Series](lesson-3.3-adql-aggregations-time-series.ipynb)